# Should Games Go on Sale? Measuring the Impact of Steam Discounts

Jasper Burkey · Senior thesis, University of Washington (advisor: Prof. Philipp Eisenhauer)

---

Four times a year Valve runs a store-wide sale, and each publisher decides whether to discount and how deep. This notebook estimates what that decision does to demand and revenue, and for which games it pays, using public data on thousands of games across 15 seasonal sales since 2023 and the machine-learning methods for causal inference surveyed in Athey and Imbens (2019).

**Contents:** 1. Data · 2. Design · 3. When the effect happens · 4. Main estimates · 5. Robustness · 6. Game size · 7. Who benefits · 8. Discount depth · 9. Conclusions and limitations

In [ ]:
import warnings

import pandas as pd
from IPython.display import display

from auxiliary import data, estimates, event_study, plots

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 90)
print(estimates.settings_note())

## 1. Data

Steam does not publish sales, so the data come from public sources, built by the scripts in [`pipeline/`](pipeline/README.md):

* **Reviews** (Steam's review API): daily review activity since January 2023, a stand-in for purchases. Exact window counts for the 645 largest games, verified to match full downloads.
* **Prices** (IsThereAnyDeal): when each game was discounted, and how deeply.
* **Sale dates:** Valve's announced seasonal sales (`pipeline/sale_calendar.csv`).
* **Game characteristics:** release dates and publishers from a daily store collector ([`collector/`](collector/README.md)).

The population is the 31,823 released, paid games with at least 50 reviews. The sample is 3,850 of them, stratified by size so that every large game is included. The unit of analysis is one game in one sale; `data/events.parquet` holds one row per game-event.

In [ ]:
events = data.load_events()
data.check_columns(events)
main_df = data.analysis_sample(events)
long_df = data.long_sample(main_df)
print(
    f"{len(events):,} game-events: {events['app_id'].nunique():,} games x {events['sale'].nunique()} sales"
)
print(
    f"Analysis sample: {len(main_df):,} game-events from {main_df['app_id'].nunique():,} games, "
    f"{main_df['discounted'].mean():.0%} discounted; {len(long_df):,} with six weeks of follow-up"
)
display(data.exclusion_table(events))
display(data.band_summary(events))

## 2. Design

Within each sale, games that discounted are compared with games that didn't, adjusting for what was known before the sale: review rate and trend, rating, age, price, a year of discount history, the response to the previous sale and the publisher's discounting habits. Comparing within the same sale removes the sale-wide surge in traffic. The identifying assumption is unconfoundedness given that pre-sale history.

**Outcomes** are log changes relative to each game's own pre-sale baseline (exp(effect) − 1 is roughly the % change): reviews, and a **revenue proxy** that values each window's reviews at the price actually paid. Since revenue is units times price, the change in log revenue is the change in log units (proxied by reviews) plus the log of the price paid relative to the regular price.

**Estimation** follows Athey and Imbens: doubly robust scores with an ensemble of LASSO, random forest and boosting, cross-fitting by game, and standard errors clustered by game. The headline, fixed in advance, is the effect for games that discounted (ATT) on the net revenue proxy.

## 3. When the effect happens

Discounted minus non-discounted games in the same sale, day by day, relative to each game's baseline. This shows the shape over time; single days have few reviews, so the size comes from section 4.

In [ ]:
curve, n_ev = event_study.event_study(events, data.load_daily())
plots.plot_event_study(curve, n_ev)
display(event_study.summarize(curve).to_frame("average difference"))

## 4. Main estimates

Four outcomes, five estimators each: the doubly robust effect for games that discounted (ATT, the headline), the average over all games (ATE), the effect for games that didn't discount (ATU), the partially linear model (PLR), and approximate residual balancing.

In [ ]:
main_table, e_main = estimates.main_estimates(main_df, long_df)
display(estimates.tidy(main_table))

**Overlap.** Discounting is highly predictable from a game's history, so many propensity scores sit near 1. The ATT needs comparable non-discounted games only where discounted games are, which is why it is the headline; the ATU relies most on extrapolation.

In [ ]:
plots.plot_overlap(e_main, main_df["discounted"].to_numpy(int))
print(
    f"Share of game-events with propensity in [0.1, 0.9]: {((e_main >= 0.1) & (e_main <= 0.9)).mean():.0%}"
)

## 5. Robustness

The log(1 + x) scale compresses changes for games with few reviews (most game-days have none), so the headline is checked on busier games and with ratio estimates, which compare mean counts directly. Winter Sales 2023 and 2024 began two to three weeks after Autumn Sales, so their baselines overlap; they are dropped as a check.

In [ ]:
display(estimates.tidy(estimates.robustness(main_df, long_df, e_main)))

## 6. Game size

Effects within each size band. For small games, use the ratio estimates (the log scale compresses their small counts); for large games, compression is negligible and the log estimates are more precise.

In [ ]:
display(estimates.tidy(estimates.by_size(long_df)))

## 7. Who benefits

An honest forest on the doubly robust scores predicts each game-event's effect; generic machine-learning inference (Chernozhukov et al.) then tests on held-out games whether the predicted differences are real (BLP), shows effects by fifth of predicted effect (GATES), and compares the most- and least-affected fifths (CLAN). Busier games are the main sample, to avoid the compression artifact; the overlap region repeats the test where discounted and non-discounted games are directly comparable.

In [ ]:
blp, gates, clan = estimates.heterogeneity(main_df)
display(blp)
plots.plot_gates(gates);

In [ ]:
keep = [
    "propensity",
    "disc_starts_365",
    "days_since_discount",
    "mean_cut_365",
    "age_days",
    "regular_usd",
    "pre_trend",
]
for (sample, outcome), t in clan.groupby(["sample", "outcome"], sort=False):
    print(f"{sample} | {outcome}: most- vs least-affected fifth")
    show = t[t["characteristic"].isin(keep)]
    display(
        show[
            [
                "characteristic",
                "most_affected",
                "least_affected",
                "difference",
                "p_adjusted",
            ]
        ].round(3)
    )

## 8. Discount depth

Each depth (1–33%, 34–60%, 61%+) versus not discounting, for all game-events (ATE) and for the game-events that chose that depth (ATT). Sale buyers often review after the sale ends, and the standard proxy values those reviews at full price; the lag-adjusted version values the two weeks after the sale at the sale price, which over-corrects, so the two bracket the truth.

In [ ]:
display(estimates.depth(main_df))

## 9. Conclusions and limitations

With the thesis settings, the estimates support six conclusions (the tables above reproduce them; with quick settings the numbers differ slightly):

1. Discounting raises review activity during the sale substantially for games that discount.
2. On the revenue proxy it costs revenue during the sale, and a modest amount net of the surrounding weeks.
3. Effects are similar across game sizes once log-scale compression is accounted for.
4. Games that discount rarely gain the most.
5. Revenue effects depend on the game: discounting pays best for younger, pricier games that discount rarely and lightly, and worst for old, cheap games that discount often and deeply.
6. For the games that use them, deep discounts bring no more activity than shallow ones but cost more revenue; shallow discounts roughly break even.

**Limitations.** Revenue is a proxy (reviews as units; DLC and in-game spending excluded). Identification assumes unconfoundedness given pre-sale history, and overlap is thin for games that rarely discount. Discounts are assumed not to affect rival games, which is untested so far. Review lag is bracketed, not resolved, and follow-up ends six weeks after each sale.

**Next steps:** validating the proxy with Steam's weekly revenue rankings, spillovers between rival games, a regression discontinuity from Steam's 30-day release rule, matrix completion as a second estimator, policy learning, and an out-of-sample replication on the Winter Sale 2026 with the collector's data.